In [1]:
##Best Practices
##Preprocessing and cleaning
##Train Test split
##BOW,TFIDF,Word2vec
##Train ML algorithms

In [2]:
##load the dataset
import pandas as pd
data=pd.read_csv('all_kindle_review.csv')


In [3]:
data.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [ ]:
##here we would require only 2 things in this usecase ,review test and raing which we are going to predict ultimately

In [5]:
data=data[['reviewText','rating']]
data.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [6]:
data.shape

(12000, 2)

In [ ]:
###Missing values->no missing value
data.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [8]:
##how many unique ratings are there
data['rating'].unique()

array([3, 5, 4, 2, 1])

In [9]:
##find out its a balanced dataset or not
data['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

In [ ]:
##Preprocessing and cleaning
##in sentiment analysis just need to find out positive rating or negative rating so any review that is greatre than 3 is +ve rating,any review that is less than 3 is a -ve rating
##less then 3 ->0 ,more than 3 ->1

In [12]:
##positive review is 1 and negative review is 0
data['rating']=data['rating'].apply(lambda x:0 if x<3 else 1)

In [13]:
df=data

In [14]:
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",1
1,Great short read. I didn't want to put it dow...,1
2,I'll start by saying this is the first of four...,1
3,Aggie is Angela Lansbury who carries pocketboo...,1
4,I did not expect this type of book to be in li...,1


In [15]:
df['rating'].unique()

array([1, 0])

In [ ]:
df['rating'].value_counts()  ##not that imbalanced ml algos the handle

rating
1    8000
0    4000
Name: count, dtype: int64

In [18]:
##text preprocessing
##1.Lower all the cases
df['reviewText']=df['reviewText'].str.lower()

In [19]:
df.head()

,reviewText,rating
0,"jace rankin may be short, but he's nothing to ...",1
1,great short read. i didn't want to put it dow...,1
2,i'll start by saying this is the first of four...,1
3,aggie is angela lansbury who carries pocketboo...,1
4,i did not expect this type of book to be in li...,1


In [ ]:
import re
import pandas as pd
from bs4 import BeautifulSoup   ##for html tag
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# make sure you have downloaded stopwords
import nltk
nltk.download('stopwords')

ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

# assuming your dataframe is df and column is 'reviewText'

# 1. Remove special characters
df['reviewText'] = df['reviewText'].apply(
    lambda x: re.sub(r'[^a-zA-Z0-9\s]', '', str(x))
)

# 2. Convert to lowercase
df['reviewText'] = df['reviewText'].apply(lambda x: x.lower())

# 3. Remove stopwords + stemming
df['reviewText'] = df['reviewText'].apply(
    lambda x: ' '.join(
        [ps.stem(word) for word in x.split() if word not in stop_words]
    )
)

# 4. Remove URLs
df['reviewText'] = df['reviewText'].apply(
    lambda x: re.sub(r'(http|https|ftp|ssh)://[^\s]+', '', x)
)

# 5. Remove HTML tags
df['reviewText'] = df['reviewText'].apply(
    lambda x: BeautifulSoup(x, 'lxml').get_text()
)

# 6. Remove extra spaces
df['reviewText'] = df['reviewText'].apply(
    lambda x: ' '.join(x.split())
)

# final cleaned text
print(df['reviewText'].head())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\KASHISH\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


0    jace rankin may short he noth mess man haul sa...
1    great short read didnt want put read one sit s...
2    ill start say first four book wasnt expect 34c...
3    aggi angela lansburi carri pocketbook instead ...
4      expect type book librari pleas find price right
Name: reviewText, dtype: object


In [21]:
df.head()

,reviewText,rating
0,jace rankin may short he noth mess man haul sa...,1
1,great short read didnt want put read one sit s...,1
2,ill start say first four book wasnt expect 34c...,1
3,aggi angela lansburi carri pocketbook instead ...,1
4,expect type book librari pleas find price right,1


In [22]:
##Lemmatizer
from nltk.stem import WordNetLemmatizer

In [23]:
lemmatizer=WordNetLemmatizer()

In [ ]:
def lemmatize_words(text):   ##any sentence given to it breaks into word lemmatizes and gives the sentence back
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

In [25]:
df['reviewText']=df['reviewText'].apply(lambda x:lemmatize_words(x))

In [26]:
df.head()

,reviewText,rating
0,jace rankin may short he noth mess man haul sa...,1
1,great short read didnt want put read one sit s...,1
2,ill start say first four book wasnt expect 34c...,1
3,aggi angela lansburi carri pocketbook instead ...,1
4,expect type book librari plea find price right,1


In [ ]:
##train test split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df['reviewText'],df['rating'],test_size=0.20)

In [28]:
##here applying bow,try using tfidf and avgword2vec

In [35]:
from sklearn.feature_extraction.text import CountVectorizer
bow=CountVectorizer()
X_train_bow=bow.fit_transform(X_train).toarray()
X_test_bow=bow.transform(X_test).toarray() #to prevent data leakage

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer()
X_train_tfidf=tfidf.fit_transform(X_train).toarray()
X_test_tfidf=tfidf.transform(X_test).toarray() #to prevent data leakage

In [37]:
X_train_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [38]:
X_train_tfidf

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [39]:
##train the ml model ->using naive bayes model

from sklearn.naive_bayes import GaussianNB
nb_model_bow=GaussianNB().fit(X_train_bow,y_train)
nb_model_tfidf=GaussianNB().fit(X_train_tfidf,y_train)

In [41]:
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report
y_pred_bow=nb_model_bow.predict(X_test_bow)
y_pred_tfidf=nb_model_tfidf.predict(X_test_tfidf)

In [42]:
acc=accuracy_score(y_test,y_pred_bow)

In [44]:
acc  ##not good accuracy try with word2vec as the dataet is huge

0.5825

In [45]:
acc=accuracy_score(y_test,y_pred_tfidf)
acc

0.5854166666666667